# Medical Diagnosis Support using Decision Trees
## Early Stage Diabetes Risk Prediction Dataset

**MDI3003 - Advanced Predictive Analytics - Lab 02**

### Educational disclaimer

This notebook is developed solely for educational and research purposes. The
resulting predictive models are **NOT** clinically validated and must **NOT** be
used for diagnosis, treatment planning, patient triage, or any real-world
healthcare decision.

# 1. Environment Setup

Import the required libraries, set display and plotting defaults, fix random
seeds for reproducibility, and record software versions. The shared engine
`meddiag_common` (`src/`) is placed on the path so the notebook, the CLI and the
GUI all use the same dataset metadata, preprocessing, metrics and persistence
code and can never disagree on protocol.

In [ ]:
# ---- 1. Environment Setup ----
# Imports, display options, random seeding for reproducibility, and a quick
# version report. A `report_figures/` dir is created for publication-quality
# 300-DPI exports used by the lab report docx.
import os
import json
import random
import warnings
import platform
from pathlib import Path
from datetime import datetime

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
import sklearn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV,
    RandomizedSearchCV, cross_validate, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, matthews_corrcoef, brier_score_loss,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance

# Persistence
import joblib

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# Visualization style
plt.style.use("ggplot")
sns.set_context("talk")

# Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("=" * 60)
print("Environment Information")
print("=" * 60)
print(f"Python Version     : {platform.python_version()}")
print(f"Pandas Version     : {pd.__version__}")
print(f"NumPy Version      : {np.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Random State       : {RANDOM_STATE}")
print("=" * 60)
# Create directory for report-quality figures
os.makedirs("report_figures", exist_ok=True)

# 2. Business Understanding and Problem Framing

## 2.1 Problem statement

Early detection of diabetes can substantially reduce long-term complications.
This notebook builds an interpretable binary classifier that predicts whether an
individual is at **positive** risk for early-stage diabetes from readily
available demographic and symptom data.

## 2.2 Prediction task

| Item | Specification |
|:---|:---|
| **Observation unit** | One individual respondent |
| **Target variable** | Diabetes risk status (`class`) |
| **Positive class** | `Positive` (diabetes risk present) -> encoded as **1** |
| **Negative class** | `Negative` (diabetes risk absent) -> encoded as **0** |
| **Prediction time** | All predictors are assumed available before a clinical work-up |
| **Intended use** | Educational predictive-analytics prototype only |
| **Out of scope** | Diagnosis, treatment recommendation, or patient triage |

## 2.3 Error priority

In a screening context, **false negatives** (missing an at-risk patient) are
typically more costly than false positives. We therefore emphasise
**sensitivity** during threshold selection while monitoring specificity and
precision.

# 3. Dataset Loading and Provenance

Load the Early Stage Diabetes Risk Prediction dataset (UCI id 529), cached at
`data/diabetes_risk_prediction.csv`. The target column `class` is binarised
(`Positive` -> 1, `Negative` -> 0); predictors are one numeric (`age`) plus 15
binary symptom/gender features.

In [ ]:
# ---- 3. Load the Early Stage Diabetes Risk dataset ----
# Read the cached CSV (relative to the notebooks/ working dir). Binarise the
# `class` target (Positive->1, Negative->0) and split into X (features) and y.
DATA_PATH = "..\data\diabetes_risk_prediction.csv"

df = pd.read_csv(DATA_PATH)

# Target encoding: Positive = 1 (disease present), Negative = 0
TARGET_COL = "class"
df[TARGET_COL] = df[TARGET_COL].map({"Positive": 1, "Negative": 0})

# Separate features and target
X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()
y.name = "DiabetesRisk"

print("=" * 60)
print("Dataset Summary")
print("=" * 60)
print(f"Rows     : {df.shape[0]}")
print(f"Columns  : {df.shape[1]}")
print(f"Features : {X.shape[1]}")
print(f"Classes  : {y.nunique()}")
print("\nTarget Mapping")
print("1 -> Positive (Diabetes risk)")
print("0 -> Negative (No diabetes risk)")
print("\nClass Distribution")
print(y.value_counts().sort_index())

# 4. Data Understanding and Audit

Audit the dataset end-to-end before any modelling: shape, dtypes, statistical
profile of `age`, missingness, duplicates, constant / quasi-constant features,
unique-value counts per column, and the target distribution.

In [ ]:
# ---- 4. Dataset understanding audit ----
# Shape, dtypes, statistical summary of age, missingness, duplicates, constant
# features, unique-value counts and the target distribution - one consolidated
# audit cell so the report has a single reproducible block of evidence.
print("=" * 60)
print("Dataset Shape")
print("=" * 60)
print(f"Rows     : {df.shape[0]}")
print(f"Columns  : {df.shape[1]}")
print(f"Features : {X.shape[1]}")
print(f"Target   : {y.name}")

display(df.head())

# Data types
print("\nData Types")
dtype_table = pd.DataFrame({
    "Feature": X.columns,
    "Data Type": X.dtypes.values
})
display(dtype_table)

# Statistical summary for age; value counts for categorical symptoms
print("\nNumerical Summary (Age)")
display(X[["age"]].describe().T)

# Missing values
missing = pd.DataFrame({
    "Missing Values": X.isna().sum(),
    "Percentage": (X.isna().mean() * 100).round(2)
})
print("\nMissing Values")
display(missing.sort_values("Missing Values", ascending=False))

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows : {duplicates}")

# Constant / quasi-constant features
constant = [c for c in X.columns if X[c].nunique() == 1]
print(f"Constant Features : {constant}")

# Unique values per column (useful for identifying categoricals)
uniq = pd.DataFrame({
    "Feature": X.columns,
    "Unique Values": X.nunique().values
})
print("\nUnique Value Counts")
display(uniq.sort_values("Unique Values"))

# Target distribution
target_counts = y.value_counts().sort_index()
target_percent = y.value_counts(normalize=True).sort_index() * 100
target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percent.round(2)
})
print("\nTarget Distribution")
display(target_summary)

## 4.1 Target distribution plot

In [ ]:
# ---- 4.1 Target distribution plot (saved for the report) ----
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(x=y, palette="Set2", ax=ax)
for container in ax.containers:
    ax.bar_label(container)
ax.set_xticklabels(["Negative", "Positive"])
ax.set_xlabel("Diabetes Risk")
ax.set_ylabel("Count")
ax.set_title("Target Class Distribution")
plt.tight_layout()
plt.savefig("report_figures/fig1_target_dist_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 4.2 Feature distributions by class

Stacked-bar crosstabs (normalised per class) for every categorical predictor,
so class imbalance does not distort the visual comparison.

In [ ]:
# ---- 4.2 Categorical-feature distributions by class (saved) ----
# Per-feature stacked-bar crosstabs (normalised by class column) so the visual
# is not distorted by the class imbalance (~62% positive).
# For categorical features, show stacked bar of Yes/No proportions by class
cat_features = [c for c in X.columns if c != "age"]

n_cols = 4
n_rows = int(np.ceil(len(cat_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, cat_features):
    crosstab = pd.crosstab(X[col], y, normalize="columns") * 100
    crosstab.plot(kind="bar", ax=ax, color=["steelblue", "coral"])
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("%")
    ax.legend(["Negative", "Positive"], title="Class")
    ax.tick_params(axis="x", rotation=0)

for ax in axes[len(cat_features):]:
    ax.remove()

plt.suptitle("Categorical Feature Distributions by Class", y=1.02, fontsize=16)
plt.tight_layout()
plt.savefig("report_figures/fig1b_categorical_distributions.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 4.3 Age distribution by class

KDE + histogram of age split by diabetes-risk class.

In [ ]:
# ---- 4.3 Age distribution by class KDE (saved) ----
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x="age", hue=y, kde=True, stat="density",
             common_norm=False, palette=["steelblue", "coral"], ax=ax)
ax.set_title("Age Distribution by Diabetes Risk")
plt.tight_layout()
plt.savefig("report_figures/fig1c_age_distribution.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 4.4 Correlation matrix

Temporarily encode the binary categoricals (Yes->1/No->0, Male->1/Female->0)
to visualise pairwise correlation. This is a visualisation-only encoding; the
modelling pipeline uses OrdinalEncoder inside the preprocessor.

In [ ]:
# ---- 4.4 Correlation matrix of encoded features (saved) ----
# Temporarily encode Yes/No and Male/Female so the heatmap is meaningful. This
# is visualisation-only; the modelling pipeline encodes inside the ColumnTransformer.
# Encode categoricals temporarily for correlation visualization
X_corr = X.copy()
for col in cat_features:
    X_corr[col] = X_corr[col].map({"Yes": 1, "No": 0, "Male": 1, "Female": 0})

corr_matrix = X_corr.corr()
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix of Features", fontsize=16)
plt.tight_layout()
plt.savefig("report_figures/fig2_correlation_matrix_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 4.5 Feature-target correlation

Rank features by absolute correlation with the binarised target to preview
which predictors are likely to appear near the top of the tree.

In [ ]:
# ---- 4.5 Feature-target correlation ranked by |r| (saved) ----
corr_target = pd.concat([X_corr, y], axis=1).corr()["DiabetesRisk"][:-1]
corr_target = corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 8))
corr_target.plot.barh()
plt.title("Feature Correlation with Target")
plt.xlabel("Correlation")
plt.tight_layout()
plt.savefig("report_figures/fig3_feature_target_corr_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 4.6 Leakage audit

All predictors are demographic/symptom data assumed available before any
diagnostic test; no post-outcome variables are present and no identifiers exist.

In [ ]:
# ---- 4.6 Leakage audit summary ----
print("=" * 60)
print("Leakage Audit")
print("=" * 60)
print(f"Identifier Columns : 0")
print(f"Duplicate Rows     : {duplicates}")
print(f"Missing Values     : {df.isna().sum().sum()}")
print("Target Leakage     : None Identified")
print("\nObservation: All predictors are demographic/symptom data assumed")
print("available before any diagnostic test. No post-outcome variables present.")

# 5. Data Preparation

Lock the stratified 80/20 train/test split before any tuning. Cross-validation
is 5-fold stratified (shuffled, seed-locked).

In [ ]:
# ---- 5. Lock the stratified 80/20 train/test split before any tuning ----
# `stratify=y` preserves the positive prevalence in both subsets; the test set
# is frozen and only touched once at the final evaluation.
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("=" * 60)
print("Dataset Split")
print("=" * 60)
print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")
print(f"Training Features: {X_train.shape[1]}")

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Rows": [len(X_train), len(X_test)],
    "Positive %": [y_train.mean() * 100, y_test.mean() * 100]
})
display(split_summary)

# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"\nCV Strategy: {cv}")

## 5.1 Preprocessing pipeline

Numeric branch: median imputation for `age`. Categorical branch: most-frequent
imputation + `OrdinalEncoder` (binary Yes/No and Male/Female are nominal for
this modelling purpose; OrdinalEncoder keeps the 0/1 form trees prefer and is
simpler to interpret downstream).

In [ ]:
# ---- 5.1 Build the preprocessing pipeline ----
# Numeric branch: median imputation for age.
# Categorical branch: most-frequent imputation + OrdinalEncoder (handles
# unknown values with -1 so production inputs never break predict).
# Identify feature types
numeric_features = ["age"]
categorical_features = [c for c in X.columns if c != "age"]  # gender + symptoms

# All categorical features are binary (Yes/No or Male/Female)
# OrdinalEncoder is sufficient and preserves interpretability for trees
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Verify preprocessor structure
print("Preprocessor constructed for:")
print(f"  Numerical features   : {numeric_features}")
print(f"  Categorical features : {len(categorical_features)}")

# 6. Reusable Evaluation Utilities

Declare the CV scoring dictionary and three reusable helpers:
`build_pipeline` (leakage-safe preprocessor + classifier), `evaluate_model`
(comprehensive metric block for a fitted model) and `cv_report` (tidy
cross-validation summary + comparison row). A generic `tune_model` supports both
`GridSearchCV` and `RandomizedSearchCV`.

In [ ]:
# ---- 6. Reusable evaluation utilities ----
# SCORING dictionary, build_pipeline, evaluate_model, cv_report, tune_model.
# Defining these once keeps every later cell concise and consistent.
SCORING = {
    "Accuracy": "accuracy",
    "Balanced_Accuracy": "balanced_accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "PR_AUC": "average_precision"
}


def build_pipeline(estimator):
    """Return a leakage-safe pipeline with preprocessing + classifier."""
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])


def evaluate_model(model, X, y):
    """Compute comprehensive metrics on a fitted model."""
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    specificity = tn / (tn + fp)
    return {
        "Accuracy": accuracy_score(y, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "Specificity": specificity,
        "F1": f1_score(y, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y, prob),
        "PR_AUC": average_precision_score(y, prob)
    }


def cv_report(model, model_name, X, y, cv, scoring, return_predictions=False):
    """Perform cross-validation and return summary + comparison row."""
    cv_results = cross_validate(
        estimator=model,
        X=X, y=y,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )
    cv_df = pd.DataFrame(cv_results)
    summary = pd.DataFrame(index=scoring.keys())
    for metric in scoring.keys():
        summary.loc[metric, "Train Mean"] = cv_df[f"train_{metric}"].mean()
        summary.loc[metric, "Train Std"] = cv_df[f"train_{metric}"].std()
        summary.loc[metric, "Validation Mean"] = cv_df[f"test_{metric}"].mean()
        summary.loc[metric, "Validation Std"] = cv_df[f"test_{metric}"].std()

    comparison_row = {
        "Model": model_name,
        "Accuracy": cv_df["test_Accuracy"].mean(),
        "Balanced_Accuracy": cv_df["test_Balanced_Accuracy"].mean(),
        "Precision": cv_df["test_Precision"].mean(),
        "Recall": cv_df["test_Recall"].mean(),
        "F1": cv_df["test_F1"].mean(),
        "ROC_AUC": cv_df["test_ROC_AUC"].mean(),
        "PR_AUC": cv_df["test_PR_AUC"].mean()
    }
    if return_predictions:
        return cv_df, summary.round(4), comparison_row
    return summary.round(4), comparison_row


def tune_model(pipeline, param_grid, X, y, cv, scoring,
               refit="ROC_AUC", n_jobs=-1, search_type="grid"):
    """
    Generic hyperparameter tuner supporting GridSearchCV or RandomizedSearchCV.
    """
    if search_type == "grid":
        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring=scoring,
            refit=refit,
            cv=cv,
            n_jobs=n_jobs,
            return_train_score=True,
            verbose=1
        )
    else:
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            scoring=scoring,
            refit=refit,
            cv=cv,
            n_jobs=n_jobs,
            return_train_score=True,
            verbose=1,
            random_state=RANDOM_STATE,
            n_iter=50
        )
    search.fit(X, y)
    return search.best_estimator_, search.best_params_, pd.DataFrame(search.cv_results_), search.best_score_

# 7. Baseline Model Development

Establish two baselines so we can verify that machine learning beats trivial
rules: a Dummy (predicts the prior) and a basic unconstrained CART. The basic
CART also reveals whether the unconstrained tree overfits.

In [ ]:
# ---- 7. Baselines: Dummy + Basic unconstrained CART ----
# Cross-validate both and inspect the basic tree's complexity to motivate the
# tuning/pruning in the next sections.
model_comparison = []

# 7.1 Dummy Baseline
dummy_pipe = build_pipeline(DummyClassifier(strategy="prior"))
dummy_summary, dummy_row = cv_report(
    dummy_pipe, "Dummy Baseline", X_train, y_train, cv, SCORING
)
model_comparison.append(dummy_row)
print("Dummy Baseline CV Summary")
display(dummy_summary)

# 7.2 Basic CART (unconstrained)
basic_cart = build_pipeline(DecisionTreeClassifier(
    criterion="gini",
    random_state=RANDOM_STATE
))
cart_summary, cart_row = cv_report(
    basic_cart, "Basic CART", X_train, y_train, cv, SCORING
)
model_comparison.append(cart_row)
print("Basic CART CV Summary")
display(cart_summary)

# Fit basic CART to inspect complexity
basic_cart.fit(X_train, y_train)
tree_basic = basic_cart.named_steps["classifier"]
print("\nBasic CART Statistics")
print(f"Tree Depth   : {tree_basic.get_depth()}")
print(f"Leaf Nodes   : {tree_basic.get_n_leaves()}")
print(f"Total Nodes  : {tree_basic.get_n_leaves() * 2 - 1}")

## 7.1 Visualise the basic CART (upper levels)

In [ ]:
# ---- 7.1 Visualise the basic CART's top 3 levels (saved) ----
# Use post-transform feature names so the tree labels are human-readable.
# Obtain feature names after preprocessing for interpretability
preprocessor.fit(X_train, y_train)
feature_names_clean = [
    f.replace("num__", "").replace("cat__", "")
    for f in preprocessor.get_feature_names_out()
]

plt.figure(figsize=(20, 10))
plot_tree(
    tree_basic,
    feature_names=feature_names_clean,
    class_names=["Negative", "Positive"],
    filled=True,
    rounded=True,
    proportion=True,
    precision=2,
    max_depth=3
)
plt.title("Basic CART (First Three Levels)")
plt.tight_layout()
plt.savefig("report_figures/fig_tree_basic_cart_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

# 8. Advanced Model Development - Pre-pruned CART

Tune the CART over pre-pruning hyperparameters (criterion, max_depth,
min_samples_split, min_samples_leaf, class_weight) with training-only CV,
refit on ROC-AUC.

In [ ]:
# ---- 8. Tune the CART over pre-pruning hyperparameters ----
# Grid over criterion / max_depth / min_samples_split / min_samples_leaf /
# class_weight; refit on ROC-AUC; training-only CV (test set untouched).
cart_pipe = build_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE))

param_grid_cart = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [3, 4, 5, 6, 8, 10, None],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__class_weight": [None, "balanced"]
}

print("Tuning pre-pruned CART...")
best_cart_estimator, best_cart_params, cart_cv_results, best_cart_score = tune_model(
    pipeline=cart_pipe,
    param_grid=param_grid_cart,
    X=X_train, y=y_train,
    cv=cv,
    scoring=SCORING,
    refit="ROC_AUC",
    search_type="grid"
)

print("\nBest Pre-pruned CART Parameters:")
print(best_cart_params)
print(f"Best CV ROC-AUC: {best_cart_score:.4f}")

# 9. Cost-Complexity Pruning

Fix the best pre-pruning parameters found above and select `ccp_alpha` by
cross-validation. Two-stage selection (pre-prune, then prune) yields the "tuned
and pruned CART".

In [ ]:
# ---- 9. Cost-complexity pruning ----
# Fit the preprocessor on the train split only, train a base tree with the best
# pre-pruning params on the transformed features, take the pruning path,
# restrict to a 20-point alpha grid (drop the trivial one-node alpha), then
# grid-search `ccp_alpha` via CV.
# Fit preprocessor on training data only
preprocessor.fit(X_train, y_train)
X_train_processed = preprocessor.transform(X_train)

# Train base tree with best pre-pruning params (without ccp_alpha)
base_tree = DecisionTreeClassifier(
    criterion=best_cart_params["classifier__criterion"],
    max_depth=best_cart_params["classifier__max_depth"],
    min_samples_split=best_cart_params["classifier__min_samples_split"],
    min_samples_leaf=best_cart_params["classifier__min_samples_leaf"],
    class_weight=best_cart_params["classifier__class_weight"],
    random_state=RANDOM_STATE
)
base_tree.fit(X_train_processed, y_train)

# Compute pruning path
path = base_tree.cost_complexity_pruning_path(X_train_processed, y_train)
ccp_alphas = path.ccp_alphas

# Exclude trivial alpha = 0 and very large values that yield a single node
ccp_alphas = ccp_alphas[ccp_alphas > 0]
if len(ccp_alphas) == 0:
    ccp_alphas = np.array([0.0])

if len(ccp_alphas) > 20:
    alpha_grid = np.linspace(ccp_alphas.min(), ccp_alphas.max(), 20)
else:
    alpha_grid = ccp_alphas

# Pipeline for pruning search
prune_pipe = build_pipeline(DecisionTreeClassifier(
    criterion=best_cart_params["classifier__criterion"],
    max_depth=best_cart_params["classifier__max_depth"],
    min_samples_split=best_cart_params["classifier__min_samples_split"],
    min_samples_leaf=best_cart_params["classifier__min_samples_leaf"],
    class_weight=best_cart_params["classifier__class_weight"],
    random_state=RANDOM_STATE
))

param_grid_alpha = {"classifier__ccp_alpha": alpha_grid}

print("Tuning ccp_alpha via cross-validation...")
best_pruned_estimator, best_alpha_params, alpha_cv_results, best_alpha_score = tune_model(
    pipeline=prune_pipe,
    param_grid=param_grid_alpha,
    X=X_train, y=y_train,
    cv=cv,
    scoring=SCORING,
    refit="ROC_AUC",
    search_type="grid"
)

best_ccp_alpha = best_alpha_params["classifier__ccp_alpha"]
print(f"\nBest ccp_alpha: {best_ccp_alpha:.6f}")
print(f"Best CV ROC-AUC after pruning: {best_alpha_score:.4f}")

## 9.1 Pruning validation curve

Plot CV ROC-AUC against `ccp_alpha` to visualise the pruning trade-off.

In [ ]:
# ---- 9.1 Pruning validation curve (CV ROC-AUC vs ccp_alpha, saved) ----
alpha_vals = alpha_cv_results["param_classifier__ccp_alpha"].astype(float).values
mean_col = next(c for c in alpha_cv_results.columns if c.startswith("mean_test_") and "roc" in c.lower())
std_col = next(c for c in alpha_cv_results.columns if c.startswith("std_test_") and "roc" in c.lower())

plt.figure(figsize=(10, 6))
plt.errorbar(
    alpha_vals,
    alpha_cv_results[mean_col].astype(float).values,
    yerr=alpha_cv_results[std_col].astype(float).values,
    fmt="o-", capsize=5
)
plt.axvline(best_ccp_alpha, color="red", linestyle="--",
            label=f"Best alpha = {best_ccp_alpha:.4f}")
plt.xlabel("ccp_alpha")
plt.ylabel("Cross-validated ROC-AUC")
plt.title("Cost-Complexity Pruning: Validation Performance vs Alpha")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("report_figures/fig4_pruning_curve_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

---

# 10. Random Forest


In [ ]:
# ---- 10. Tune a Random Forest via randomised search ----
# Broad space over n_estimators / max_depth / min_samples_split /
# min_samples_leaf / max_features / class_weight; 50 draws; refit on ROC-AUC.
rf_pipe = build_pipeline(RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=4
))

param_dist_rf = {
    "classifier__n_estimators": [100, 300, 500, 700],
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2", 0.3, 0.5],
    "classifier__class_weight": [None, "balanced"]
}

print("Tuning Random Forest with RandomizedSearchCV...")
best_rf_estimator, best_rf_params, rf_cv_results, best_rf_score = tune_model(
    pipeline=rf_pipe,
    param_grid=param_dist_rf,
    X=X_train, y=y_train,
    cv=cv,
    scoring=SCORING,
    refit="ROC_AUC",
    search_type="random"
)

print("\nBest Random Forest Parameters:")
print(best_rf_params)
print(f"Best CV ROC-AUC: {best_rf_score:.4f}")

# 11. Model Comparison (Cross-Validation)

Compare the four models - Dummy, Basic CART, Tuned & Pruned CART, Random
Forest - under identical 5-fold stratified CV folds.

## 11.1 Split-criterion comparison

Quick Gini-vs-Entropy comparison on a basic CART to confirm criterion choice
does not meaningfully change discrimination.

In [ ]:
# ---- 11.1 Split-criterion comparison: gini vs entropy on a basic CART ----
criterion_comparison = []
for crit in ["gini", "entropy"]:
    pipe = build_pipeline(DecisionTreeClassifier(criterion=crit, random_state=RANDOM_STATE))
    crit_summary, crit_row = cv_report(pipe, f"Basic CART ({crit})", X_train, y_train, cv, SCORING)
    criterion_comparison.append(crit_row)

crit_df = pd.DataFrame(criterion_comparison)
crit_df = crit_df.sort_values("ROC_AUC", ascending=False).round(4)
print("Split Criterion Comparison")
display(crit_df)

In [ ]:
# ---- 11. Re-evaluate all four models under identical CV folds ----
# Reuse cv_report so the comparison is apples to apples; sort by CV ROC-AUC.
models_to_compare = {
    "Dummy Baseline": build_pipeline(DummyClassifier(strategy="prior")),
    "Basic CART": build_pipeline(DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE
    )),
    "Tuned & Pruned CART": best_pruned_estimator,
    "Random Forest": best_rf_estimator
}

comparison_rows = []
comparison_summaries = {}

for name, model in models_to_compare.items():
    print(f"Evaluating {name}...")
    summary, row = cv_report(model, name, X_train, y_train, cv, SCORING)
    comparison_summaries[name] = summary
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values("ROC_AUC", ascending=False).round(4)

print("\nCross-Validation Model Comparison")
display(comparison_df)

# 12. Operating Threshold Selection

Sweep the operating threshold on out-of-fold training predictions of the chosen
final model (here the Tuned & Pruned CART for interpretability, with Random
Forest available as a higher-performance alternative). Enforce sensitivity >=
0.95 and pick the highest-specificity threshold that still meets the target;
fall back to max-Youden if the target is infeasible.

## 12.1 Model-comparison visualisation

Bar chart of the four models across the key cross-validated metrics
(Accuracy, F1, Balanced Accuracy, ROC_AUC, PR_AUC).

In [ ]:
# ---- 12.1 Bar chart of the four models across the key CV metrics (saved) ----
plot_metrics = ["Accuracy", "F1", "Balanced_Accuracy", "ROC_AUC", "PR_AUC"]

x = np.arange(len(plot_metrics))
width = 0.18
multiplier = 0

fig, ax = plt.subplots(figsize=(14, 8))

for i, row in comparison_df.iterrows():
    offset = width * multiplier
    vals = [row[m] for m in plot_metrics]
    bars = ax.bar(x + offset, vals, width, label=row["Model"])
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=0)
    multiplier += 1

ax.set_xlabel("Evaluation Metric", fontsize=12, fontweight="bold")
ax.set_ylabel("Cross-Validated Score", fontsize=12, fontweight="bold")
ax.set_title("Comprehensive Model Comparison: Tuned CART, Random Forest, and Baselines", fontsize=14, fontweight="bold")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([m.replace("_", " ") for m in plot_metrics], fontsize=10)
ax.legend(loc="lower right", fontsize=10, title="Models")
ax.set_ylim(0, 1.15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("report_figures/fig5_model_comparison_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

In [ ]:
# ---- 12. Sweep the operating threshold on out-of-fold training predictions ----
# Generate OOF probabilities via cross_val_predict (each train row predicted by
# a fold that never saw it). Sweep thresholds; enforce sensitivity >= 0.95 and
# pick the highest-specificity threshold that still meets the target; fall back
# to max Youden if the target is infeasible.
# For interpretability we select the Tuned & Pruned CART as the final model,
# per the lab manual guidance. Random Forest may be substituted if raw
# performance is the sole priority.
final_model = best_pruned_estimator

oof_proba = cross_val_predict(
    final_model,
    X_train, y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

thresholds = np.linspace(0.01, 0.99, 199)
threshold_metrics = []

for thresh in thresholds:
    pred = (oof_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = (2 * precision * sensitivity / (precision + sensitivity)
          if (precision + sensitivity) > 0 else 0)
    threshold_metrics.append({
        "threshold": thresh,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1": f1,
        "youden": sensitivity + specificity - 1
    })

threshold_df = pd.DataFrame(threshold_metrics)

# Strategy 1: Maximize Youden's index
best_youden_idx = threshold_df["youden"].idxmax()
selected_threshold = threshold_df.loc[best_youden_idx, "threshold"]
print(f"Selected threshold (max Youden): {selected_threshold:.3f}")

# Strategy 2: Enforce minimum sensitivity (educational constraint)
TARGET_SENSITIVITY = 0.95
feasible = threshold_df[threshold_df["sensitivity"] >= TARGET_SENSITIVITY]
if not feasible.empty:
    selected_threshold = feasible.sort_values(
        "specificity", ascending=False
    ).iloc[0]["threshold"]
    print(f"Selected threshold (sensitivity >= {TARGET_SENSITIVITY}): {selected_threshold:.3f}")
else:
    print("No threshold meets target sensitivity; using max Youden.")

## 12.2 Threshold plot

Sensitivity and specificity curves against the operating threshold, with the
selected threshold marked.

In [ ]:
# ---- 12.2 Plot sensitivity/specificity vs threshold (saved) ----
plt.figure(figsize=(10, 6))
plt.plot(threshold_df["threshold"], threshold_df["sensitivity"],
         label="Sensitivity", linewidth=2)
plt.plot(threshold_df["threshold"], threshold_df["specificity"],
         label="Specificity", linewidth=2)
plt.axvline(selected_threshold, color="red", linestyle="--",
            label=f"Selected = {selected_threshold:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Metric")
plt.title("Sensitivity and Specificity vs Threshold (Out-of-Fold)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("report_figures/fig6_threshold_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

# 13. Final Evaluation on the Locked Test Set

Lock the model on all training data and evaluate on the test set exactly once.
Report sensitivity, specificity, precision, NPV, F1, balanced accuracy, ROC-AUC,
PR-AUC, Brier, MCC and the explicit TN/FP/FN/TP at the chosen threshold.

In [ ]:
# ---- 13. Lock the final model on all train data; score the test set ONCE ----
# `evaluate_binary` builds the full metric block used in the report.
final_model.fit(X_train, y_train)
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)


def evaluate_binary(y_true, y_pred, y_proba, threshold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    return {
        "threshold": threshold,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "precision_ppv": precision_score(y_true, y_pred, zero_division=0),
        "npv": npv,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "brier_score": brier_score_loss(y_true, y_proba)
    }


test_metrics = evaluate_binary(y_test, test_pred, test_proba, selected_threshold)
test_metrics_df = pd.DataFrame([test_metrics]).round(4)
print("Final Test Metrics")
display(test_metrics_df)

## 13.1 Confusion matrix

In [ ]:
# ---- 13.1 Test-set confusion matrix (saved) ----
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred,
    display_labels=["Negative", "Positive"],
    cmap="Blues",
    values_format="d",
    ax=ax
)
ax.set_title(f"Test Confusion Matrix (threshold = {selected_threshold:.3f})")
plt.tight_layout()
plt.savefig("report_figures/fig7_confusion_matrix_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 13.2 ROC curve

In [ ]:
# ---- 13.2 ROC curve (saved) ----
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, test_proba, name="Final Model", ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Chance")
ax.set_title("ROC Curve - Test Set")
ax.legend()
plt.tight_layout()
plt.savefig("report_figures/fig8_roc_curve_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 13.3 Precision-recall curve

In [ ]:
# ---- 13.3 Precision-recall curve (saved) ----
fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_test, test_proba, name="Final Model", ax=ax)
ax.axhline(y_test.mean(), color="gray", linestyle="--", label="Positive prevalence")
ax.set_title("Precision-Recall Curve - Test Set")
ax.legend()
plt.tight_layout()
plt.savefig("report_figures/fig9_pr_curve_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 13.4 Calibration plot (reliability diagram)

In [ ]:
# ---- 13.4 Calibration plot / reliability diagram (saved) ----
from sklearn.calibration import calibration_curve

def plot_calibration(y_true, prob, model_name, ax, color):
    prob_true, prob_pred = calibration_curve(y_true, prob, n_bins=10, strategy="uniform")
    ax.plot(prob_pred, prob_true, marker="o", linewidth=2, label=model_name, color=color)

fig, ax = plt.subplots(figsize=(9, 8))
plot_calibration(y_test, test_proba, f"Final Model (Brier={test_metrics['brier_score']:.3f})", ax, "blue")
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.6)
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Fraction of Positives")
ax.set_title("Calibration Plot (Reliability Diagram)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.savefig("report_figures/fig9b_calibration_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

# 14. Interpretation and Feature Importance

Inspect the fitted tree's complexity, visualise the pruned tree, export its
if-then rules, and quantify importance both by impurity (tree-internal) and by
permutation (model-agnostic, on the test set).

In [ ]:
# ---- 14. Inspect the fitted tree complexity (depth / leaves / nodes) ----
# Extract the fitted tree from the final pipeline
fitted_tree = final_model.named_steps["classifier"]

print("Final Pruned Tree Complexity")
print(f"Depth       : {fitted_tree.get_depth()}")
print(f"Leaves      : {fitted_tree.get_n_leaves()}")
print(f"Nodes       : {fitted_tree.get_n_leaves() * 2 - 1}")

## 14.1 Visualise the final pruned tree (top 4 levels)

In [ ]:
# ---- 14.1 Visualise the final pruned tree (top 4 levels, saved) ----
plt.figure(figsize=(20, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names_clean,
    class_names=["Negative", "Positive"],
    filled=True,
    rounded=True,
    proportion=True,
    precision=2,
    max_depth=4  # readable upper levels
)
plt.title("Final Tuned & Pruned CART")
plt.tight_layout()
plt.savefig("report_figures/fig_tree_final_cart_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 14.2 Decision rules (text)

In [ ]:
# ---- 14.2 Export the final tree's if-then rules as text (truncated) ----
rules = export_text(fitted_tree, feature_names=feature_names_clean)
print(rules[:5000])

## 14.3 Impurity-based feature importance

In [ ]:
# ---- 14.3 Impurity-based feature importance + bar chart (saved) ----
# Non-causal importance biased toward high-cardinality / many-split features.
impurity_importance = pd.Series(
    fitted_tree.feature_importances_,
    index=feature_names_clean
).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
impurity_importance.head(15).plot.barh()
plt.title("Top 15 Feature Importances (Impurity-Based)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("report_figures/fig10_feature_importance_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

print("\nImpurity Importance (top 10)")
print(impurity_importance.head(10))

## 14.4 Permutation importance (test set)

In [ ]:
# ---- 14.4 Permutation importance on the test set (model-agnostic, saved) ----
# Measures how much ROC-AUC drops when each feature is randomly shuffled.
perm = permutation_importance(
    final_model, X_test, y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": feature_names_clean,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print("Permutation Importance (top 10)")
display(perm_df.head(10))

plt.figure(figsize=(10, 8))
perm_df.head(15).set_index("feature")["importance_mean"].plot.barh(
    xerr=perm_df.head(15).set_index("feature")["importance_std"]
)
plt.title("Permutation Importance (Test Set, ROC-AUC)")
plt.xlabel("Drop in ROC-AUC")
plt.tight_layout()
plt.savefig("report_figures/fig10b_permutation_importance_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

## 14.5 Decision-path tracing

Trace a hand-picked sample (a correct and an incorrect prediction where
available) through the tree, printing each node's split, threshold and leaf
class proportion. This is the lab's interpretability evidence.

In [ ]:
# ---- 14.5 Trace two samples (one correct, one wrong where available) ----
# Print the per-node split predicate + leaf class proportion to show *why* the
# tree decided the way it did for concrete records.
test_results_df = X_test.copy()
test_results_df["true_label"] = y_test.values
test_results_df["predicted"] = test_pred
test_results_df["probability"] = test_proba
test_results_df["correct"] = (y_test.values == test_pred)

correct_samples = test_results_df[test_results_df["correct"]].sample(2, random_state=RANDOM_STATE)
incorrect_count = (~test_results_df["correct"]).sum()
if incorrect_count > 0:
    incorrect_samples = test_results_df[~test_results_df["correct"]].sample(
        min(2, incorrect_count), random_state=RANDOM_STATE
    )
    trace_samples = pd.concat([correct_samples, incorrect_samples])
else:
    trace_samples = correct_samples

def trace_decision_path(model, sample_row, feature_names):
    """
    Trace the decision path for one sample after preprocessing.
    """
    preprocessor = model.named_steps["preprocessor"]
    tree = model.named_steps["classifier"]

    # Convert the sample row into a single-row DataFrame with the original feature columns
    sample_df = pd.DataFrame([sample_row.values], columns=X.columns)

    # Transform using the fitted preprocessor
    sample_array = preprocessor.transform(sample_df)

    # Get path as a sparse matrix and flatten to node indices
    path_matrix = tree.decision_path(sample_array)
    node_ids = np.flatnonzero(path_matrix.toarray().ravel())

    feature = tree.tree_.feature
    threshold = tree.tree_.threshold
    n_node_samples = tree.tree_.n_node_samples
    value = tree.tree_.value

    rules_list = []
    for node in node_ids:
        if feature[node] != -2:
            feat_name = feature_names[feature[node]]
            sample_val = sample_array[0, feature[node]]
            direction = "<=" if sample_val <= threshold[node] else ">"
            rules_list.append(
                f"  Node {node}: {feat_name} ({sample_val:.2f}) {direction} {threshold[node]:.2f}"
            )
        else:
            n_pos = int(value[node][0, 1])
            n_neg = int(value[node][0, 0])
            n_total = int(n_node_samples[node])
            frac = n_pos / n_total if n_total > 0 else 0
            rules_list.append(
                f"  Leaf {node}: n={n_total}, positive={n_pos} ({frac:.1%}), negative={n_neg}"
            )
    return rules_list

print("=" * 60)
print("Decision Path Tracing for Individual Samples")
print("=" * 60)

for idx, (sample_idx, sample) in enumerate(trace_samples.iterrows()):
    print()
    print("=" * 60)
    print(f"Sample {idx + 1} (Index {sample_idx})")
    print("=" * 60)

    true_label = "Positive" if sample["true_label"] == 1 else "Negative"
    pred_label = "Positive" if sample["predicted"] == 1 else "Negative"
    correctness = "CORRECT" if sample["correct"] else "INCORRECT"

    print(f"True Label     : {true_label}")
    print(f"Predicted      : {pred_label}")
    print(f"Probability    : {sample['probability']:.4f}")
    print(f"Prediction     : {correctness}")
    print()

    print("Decision Path:")
    sample_for_path = sample.drop(
        ["true_label", "predicted", "probability", "correct"],
        errors="ignore"
    )
    for rule in trace_decision_path(final_model, sample_for_path, feature_names_clean):
        print(rule)

## 14.6 Robustness analysis

Re-fit the basic tree across 10 random train/test seeds and report the mean and
standard deviation of CV ROC-AUC - a lightweight split-stability check.

# 15. Save Artifacts

Persist every artefact the CLI/GUI need - the fitted final pipeline, the
operating threshold, individual model pipelines, the CV and test results CSVs
and the metadata JSON - via the shared  engine,
so the notebook and the CLI can never disagree. A legacy
 bundle is also preserved for backwards
compatibility.

In [ ]:
# ---- 14.6 Robustness analysis: split stability across 10 random seeds ----
N_REPEATS = 10
stability_results = []

for i in range(N_REPEATS):
    seed = i * 10
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    cv_stable = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    stable_pipe = build_pipeline(DecisionTreeClassifier(random_state=seed))
    stable_pipe.fit(X_tr, y_tr)
    stable_scores = cross_validate(stable_pipe, X_tr, y_tr, cv=cv_stable, scoring=SCORING, n_jobs=-1)
    stability_results.append({
        "seed": seed,
        "model": "Tuned & Pruned CART",
        "roc_auc": stable_scores["test_ROC_AUC"].mean(),
        "sensitivity": stable_scores["test_Recall"].mean(),
        "specificity": stable_scores["test_Specificity"].mean() if "test_Specificity" in stable_scores else None
    })

stability_df = pd.DataFrame(stability_results)
print("=" * 60)
print("Split Stability Analysis (10 Random Seeds)")
print("=" * 60)
display(stability_df.groupby("model").agg({"roc_auc": ["mean", "std"], "sensitivity": ["mean", "std"]}).round(4))

fig, ax = plt.subplots(figsize=(10, 5))
for model_name in stability_df["model"].unique():
    subset = stability_df[stability_df["model"] == model_name]
    ax.plot(subset["seed"], subset["roc_auc"], marker="o", linestyle="-", label=model_name)
ax.set_xlabel("Random Seed")
ax.set_ylabel("CV ROC-AUC")
ax.set_title("Model Stability Across Different Data Splits")
ax.legend(fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.savefig("report_figures/fig_robustness_dia.png", dpi=300, bbox_inches="tight")
# plt.show()  # commented: figure saved for report

In [ ]:
# ---- 15. Persist artefacts via the shared meddiag_common engine ----
# Build a minimal `state` dict from the objects already trained in this
# notebook and call `meddiag_common.save_artifacts` so the CLI/GUI can load the
# saved pipeline, threshold and metadata from `artifacts/diabetes_*` without
# retraining. A legacy `diabetes_full_artifact_bundle.joblib` is also preserved.
import joblib  # noqa: E402

# Map the notebook's trained estimators into the shared-registry naming.
_models_registry = {
    "Dummy (prior)": build_pipeline(DummyClassifier(strategy="prior")),
    "Basic CART": build_pipeline(DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE)),
    "Tuned and pruned CART": best_pruned_estimator,
    "Random Forest": best_rf_estimator,
    "Logistic Regression": build_pipeline(
        __import__("sklearn.linear_model",
                   fromlist=["LogisticRegression"]).LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
}
# Fit any unfitted pipeline in the registry on the full train split so that
# every saved .joblib is ready-to-predict, matching the CLI's expectations.
for _name, _pipe in _models_registry.items():
    _inner = _pipe.named_steps.get("classifier") if hasattr(_pipe, "named_steps") else _pipe
    if not getattr(_inner, "classes_", None):
        _pipe.fit(X_train, y_train)

_state = {
    "spec": M.spec("diabetes"),
    "X": df.drop(columns=[TARGET_COL]), "y": y,
    "X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
    "cv": cv,
    "models": _models_registry,
    "pruned_gs1": None, "pruned_gs2": None,
    "pruned_params": {f"model__{k}": v for k, v in best_cart_params.items()
                     if k != "model__ccp_alpha"} if "best_cart_params" in dir() else {},
    "rf_gs": None, "rf_params": best_rf_params if "best_rf_params" in dir() else {},
    "cv_table": comparison_df.rename(columns={
        "Accuracy": "accuracy", "Balanced_Accuracy": "balanced_accuracy",
        "Precision": "precision", "Recall": "recall", "F1": "f1",
        "ROC_AUC": "roc_auc", "PR_AUC": "pr_auc",
    }) if "comparison_df" in dir() else pd.DataFrame(
        [{"Model": name, "roc_auc_mean": 0} for name in _models_registry]),
    "oof_prob": None, "threshold_df": threshold_df if "threshold_df" in dir() else pd.DataFrame(),
    "threshold": float(selected_threshold),
    "threshold_target_satisfied": float(
        threshold_df.loc[threshold_df.threshold <= selected_threshold, "sensitivity"].max()
    ) if "threshold_df" in dir() and len(threshold_df) else 0.95,
    "best_name": "Tuned and pruned CART" if final_model is best_pruned_estimator
                 else "Random Forest",
    "best_pipe": final_model,
    "test_metrics": {**test_metrics,
                     **{"brier": test_metrics.get("brier_score",
                                                  test_metrics.get("brier", 0.0))}},
    "default_threshold_metrics": {**test_metrics},
    "all_test_rows": [dict(
        Model="Tuned and pruned CART" if final_model is best_pruned_estimator
              else "Random Forest", **test_metrics)],
    "test_prob": test_proba, "test_pred": test_pred,
    "feature_cols": list(X.columns), "class_names": ("Negative", "Positive"),
    "numeric_cols": numeric_features, "categorical_cols": categorical_features,
    "complexity": {},
}

_paths = M.save_artifacts("diabetes", _state, tag="diabetes")
print("Artifacts saved (CLI/GUI compatible) to:", os.path.abspath(M.ART_DIR))
print("Metadata:", _paths.get("metadata"))
artifacts_dir = Path("artifacts_diabetes")
artifacts_dir.mkdir(exist_ok=True)

artifact_bundle = {
    "preprocessor": preprocessor,
    "dummy_pipeline": build_pipeline(DummyClassifier(strategy="prior")),
    "basic_cart_pipeline": build_pipeline(DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE
    )),
    "best_cart_pipeline": best_cart_estimator,
    "pruned_cart_pipeline": best_pruned_estimator,
    "random_forest_pipeline": rf_pipe,
    "best_random_forest_pipeline": best_rf_estimator,
    "final_model": final_model,
    "threshold": float(selected_threshold),
    "positive_class": "Positive (Diabetes risk)",
    "feature_names": feature_names_clean,
    "random_state": RANDOM_STATE,
    "test_metrics": test_metrics,
    "software": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__
    },
    "intended_use": "Educational predictive-analytics laboratory only",
    "prohibited_use": "Clinical diagnosis, treatment, triage, or patient management",
    "dataset_source": "UCI ML Repository - Early Stage Diabetes Risk Prediction",
    "date_created": datetime.now().isoformat()
}

# Save full bundle
bundle_path = artifacts_dir / "diabetes_full_artifact_bundle.joblib"
joblib.dump(artifact_bundle, bundle_path)

# Save individual fitted objects
for name, obj in artifact_bundle.items():
    if isinstance(obj, (dict, list, str, int, float, bool)):
        continue
    joblib.dump(obj, artifacts_dir / f"{name}.joblib")

# Save metrics as CSV
pd.DataFrame([test_metrics]).to_csv(
    artifacts_dir / "diabetes_test_metrics.csv", index=False
)

# Save CV comparison
comparison_df.to_csv(
    artifacts_dir / "diabetes_cv_comparison.csv", index=False
)

print("Artifacts saved to:", artifacts_dir.resolve())
print("Bundle file:", bundle_path.resolve())

# 16. Conclusion and Responsible Reporting

## 16.1 Summary of findings

1. **Baseline vs. models** - Both the Tuned & Pruned CART and the Random Forest
   substantially outperform the Dummy baseline across all discrimination metrics.
2. **Pruning effect** - Pre-pruning and cost-complexity pruning reduced tree
   depth and leaf count compared with the basic CART, improving validation
   stability while maintaining strong sensitivity.
3. **Error analysis** - Final test evaluation at the selected operating
   threshold yields explicit TN/FP/FN/TP counts. False negatives are minimised
   given the screening-oriented threshold strategy.
4. **Interpretability** - The pruned tree exposes explicit if-then rules
   involving age, polyuria, gender and other symptoms. These are
   **dataset-derived thresholds**, not clinically validated cut-offs.
5. **Limitations** - small, geographically limited sample; self-reported symptom
   data subject to recall and selection bias; no external validation cohort
   (transportability unknown); feature importance describes *association*, not
   *causation*.

## 16.2 Requirements before real-world use

- Prospective clinical validation on an independent, representative cohort.
- Calibration assessment and subgroup (age, sex) performance audits.
- Governance framework including human oversight, periodic retraining and a
  rollback protocol.
- Regulatory review if ever intended for any diagnostic or triage workflow.